# Reproducing Martinez-Conde et al. (2019) from CRCNS pvc-12

**Paper.** Martinez-Conde S, McCamy MB, Troncoso XG, Otero-Millan J, Macknik SL (2019). Area V1
responses to illusory corner-folds in Vasarely's nested squares and the Alternating Brightness Star
illusions. *PLoS ONE* 14(3): e0210941.
[doi:10.1371/journal.pone.0210941](https://doi.org/10.1371/journal.pone.0210941)

**Dataset.** CRCNS pvc-12, "Single-neuron extracellular recordings from area V1 of awake behaving
rhesus monkeys viewing brightness illusions based on Victor Vasarely's artworks",
[doi:10.6080/K05D8Q1T](https://doi.org/10.6080/K05D8Q1T). Read in place from the public S3 bucket
`crcnsarchive` (`pvc-12/live/V1_Vasarely_data.zip`), never written to disk.

**Version.** CRCNS data sets are not versioned, so the version is recorded as `unversioned`
(docs/archives.md, "CRCNS"); there is only one published copy to choose, the `live` folder, which
the paper's data statement points to through the DOI. To make a later change detectable, the
notebook prints the data set's `checksums.md5` and checks that the zip's ETag on the bucket equals
its MD5 there, `e1301fe463e72332e2303b2053ee0559` (the zip was last modified 2026-02-11). The data
descriptor read is version 0.7 (Feb 15, 2018).

**Claims.** `claims/martinez-conde-2019.yaml`, written from the version of record (PMC6438452,
Europe PMC full text: main text, Methods, legends, Table 1 and the supplementary legends; S1 and
S2 Tables themselves were not available).

| Claim | Tested or blocked |
|---|---|
| neuron-counts | tested |
| spikes-per-condition | blocked: the spike count and eye time maps are not in the files |
| s4-baseline-no-difference | tested |
| fig4a-above-baseline | tested |
| fig4b-spike-rate-difference | tested |
| fig4b-pairwise-significance | tested |
| fig5-average-maps | tested |
| table1-edge, table1-p180, table1-s90, table1-s52, table1-s40 | tested |
| fig6-midpoint-representative | tested |
| fig7a-rms-contrast-fit | tested |
| fig7c-fold-edge-equivalence | tested |
| fig7c-fold-edge-significance | tested |
| fig7d-nonlinear | tested |
| fig7d-dog-similar | blocked: the DOG model output of reference [3] is not public as data |
| s3-latency-peak | blocked: no spike times or eye positions, only maps made at 60 ms |

**Questions answered.** This attempt asked a reader (`questions/martinez-conde-2019.yaml`) whether
the Corners S40 response of fig4b-spike-rate-difference and table1-s40 should be taken at the
bright S40 fold alone, as the Methods say only bright folds were analysed, in place of the two
folds averaged or "the fold at the ROI centre" (neither fold is at the ROI centre). The reader
amended the claims file: its opening comment and the criteria of fig4a-above-baseline,
fig4b-spike-rate-difference, fig4b-pairwise-significance and table1-s40 now take the bright S40
fold alone. This notebook follows the amended file; the two folds averaged are printed only as a
reference in fig4b-spike-rate-difference and Table 1, to show what the answer changed.

Every Star result rests on the 12 of 27 star recordings whose files hold the authors' aligned map
(see the data check); the paper has 30 Star neurons.

**Revision after verification round 1.** The verifier's report
(`transcripts/martinez-conde-2019/verifier-1-report.md`) led to these changes. The baseline
placement is now chosen per stimulus by comparison with the files' `BaseActivity`, shown file by
file in the data check, with two more Corners placements and the main reading on the files that
have `BaseActivity`. Revising it showed that the Square and Edge squares centred on the corners
had run past the authors' frame, where the helper had read zeros; they are now read from the
screen maps. The Edge measures use the 18 files whose Edge is wholly on the screen. The
registration check had failed for three Corners files through an error in unwrapping the
correlation lag; it is fixed and all 60 files register. Blocks of non-integer length (S90) are
now sampled symmetrically. The Fig 7C margin, the 1% Edge profile and the units check are printed.

In [ ]:
import time
import resource
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import ndimage, stats
from statsmodels.stats.multitest import multipletests

import warnings

import vasarely as v

warnings.filterwarnings("ignore", category=RuntimeWarning)
T0 = time.time()
DATASET = "crcns:pvc-12/unversioned"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
print(DATASET)
print(urllib.request.urlopen(v.CHECKSUMS_URL).read().decode())
head = urllib.request.urlopen(urllib.request.Request(v.URL, method="HEAD"))
etag = head.headers["ETag"].strip('"')
print("zip ETag", etag, "Last-Modified", head.headers["Last-Modified"], "size", head.headers["Content-Length"])
assert etag == v.ZIP_MD5, "the zip differs from the copy this notebook was written on"

## Data check

In [ ]:
z = v.zip_file()
members = [i for i in z.infolist() if not i.is_dir()]
print(len(members), "files in the zip,", round(sum(i.file_size for i in members) / 1e6, 1), "MB uncompressed")
files = pd.DataFrame(v.list_files())
print(files.groupby(["folder", "prefix"]).size().to_string())
recs = [v.load_record(r) for r in files.to_dict("records")]
print("loaded in", round(time.time() - T0), "s; peak memory", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")
info = pd.DataFrame([{"code": r["code"], "folder": r["folder"], "cell": r["cell"], "monkey": r["monkey"],
                      "stimulus_type": r["stim_type"], "mat": r["mat"] + "/" + r["stim_mat"],
                      "standard": r["has_standard"], "BaseActivity": np.isfinite(r.get("base_activity", np.nan)),
                      "fields": " ".join(r["fields"])} for r in recs])
print(info.groupby(["folder", "stimulus_type", "standard"]).size().to_string())
print()
print(info.groupby(["folder", "fields"]).size().to_string())

In [ ]:
# smoothed_image_norm: which files hold it, and what it is against smoothed_image of the same file
has_norm = info.fields.str.split().apply(lambda f: "smoothed_image_norm" in f)
print("smoothed_image_norm present in", int(has_norm.sum()), "of", len(info), "files, absent from", int((~has_norm).sum()))
kinds = []
for r in files.to_dict("records"):
    kind_, d = v.load_mat(r["path"])
    if kind_ == "h5" or "smoothed_image_norm" not in d.dtype.names:
        continue
    n_, s_ = v._arr(d["smoothed_image_norm"]), v._arr(d["smoothed_image"])
    S = d["Standard"][0, 0]
    base_ = float(np.asarray(S["BaseActivity"]).ravel()[0]) if "BaseActivity" in S.dtype.names else np.nan
    if not np.isfinite(n_).all():
        kinds.append({"code": r["code"], "kind": "holds non-finite values"}); continue
    A = np.c_[s_.ravel(), np.ones(s_.size)]
    (slope_, icpt_), *_ = np.linalg.lstsq(A, n_.ravel(), rcond=None)
    resid_ = np.abs(A @ [slope_, icpt_] - n_.ravel()).max()
    if resid_ < 1e-9 and abs(slope_ - 1) < 1e-9:
        k_ = "smoothed_image minus a constant" + (" (= BaseActivity)" if np.isfinite(base_) and abs(-icpt_ - base_) < 1e-12 else "")
    elif resid_ < 1e-9:
        k_ = "linear rescaling of smoothed_image"
    else:
        k_ = "not a linear function of smoothed_image"
    kinds.append({"code": r["code"], "kind": k_, "min": n_.min(), "max": n_.max(), "pixels at -1": (n_ == -1).mean()})
kinds = pd.DataFrame(kinds)
print(kinds.groupby("kind").agg(files=("code", "size"), min_low=("min", "min"), min_high=("min", "max"), max_low=("max", "min"),
                                max_high=("max", "max"), at_minus1_low=("pixels at -1", "min"), at_minus1_high=("pixels at -1", "max")).round(3).to_string())


The descriptor says each spike rate map file holds `image` (raw spike counts),
`smoothed_image` and `smoothed_image_norm` (normalised by eye time). The files differ: the fields
vary between files, and most hold a `Standard` struct and a `norm` flag the descriptor does not
mention. `smoothed_image_norm` is absent from 39 of 77 files. In the 38 that hold it, it is
`smoothed_image` minus the file's `BaseActivity` in 5 (among them ABC050900J1A, the one file with
an eye time map, where it is therefore not normalised by eye time), a linear rescaling of
`smoothed_image` with a maximum of 1.0 to 2.9 in 21, and in 11 not a linear function of
`smoothed_image`, with 26% to 77% of the pixels at exactly -1 and the rest below 1; in one it holds
non-finite values. Two files in the `Star` folder (`ABS042700J2`, `ABS051000J3`) are MATLAB 7.3
files whose stimulus metadata names the stimulus type `CORNERS`; they are checked below.

In [ ]:
# The spreadsheet that lists neurons and conditions
xl = pd.read_excel(v.zip_file().open(v.XLSX), header=None)
hdr = [i for i in range(len(xl)) if str(xl.iloc[i, 2]) == "Cell"][0]
sheet = xl.iloc[hdr + 1:, 2:6].copy()
sheet.columns = ["cell", "Corners", "Star", "Edge-Compare"]
sheet["cell"] = sheet["cell"].astype(str)
marks = sheet[["Corners", "Star", "Edge-Compare"]].notna()
print("rows in the spreadsheet:", len(sheet), "; marked conditions per column:", marks.sum().to_dict())
print("rows with no condition marked:", sheet.loc[~marks.any(axis=1), "cell"].tolist())
folder_of = {"Corners": "Corners", "Star": "Star", "Edge-Compare": "Edge-compare"}
in_files = {(r["cell"], r["folder"]) for r in recs}
mism = [(c, k) for c, row in zip(sheet["cell"], marks.to_dict("records")) for k, m in row.items()
        if m != ((c, folder_of[k]) in in_files)]
print("spreadsheet marks that do not match a file:", mism)
cells_with_files = sorted({r["cell"] for r in recs})
print("neurons with files:", len(cells_with_files), "; by monkey:",
      pd.Series([c[-2] for c in cells_with_files]).value_counts().to_dict())

In [ ]:
# The two MATLAB 7.3 files in the Star folder: stimulus metadata and comparison with the Corners file of the same cell
for code_ in ["ABS042700J2", "ABS051000J3"]:
    g = v.load_mat(f"{v.ROOT}Star/{code_}_Stimulus.mat")[1]
    stim = g["data"]["stim"]
    print(code_, "type", "".join(chr(c) for c in np.array(g["type"]).ravel()),
          {k: float(np.array(stim[k]).ravel()[0]) for k in ["size", "depth", "angle", "width", "grades"]})
scr = {r["code"]: v.load_record(r, with_screen=True)["screen_smoothed"]
       for r in files.to_dict("records") if r["cell"] in ("042700J2", "051000J3")}
for a, b in [("ABS042700J2", "ABC042700J2"), ("ABS051000J3", "ABC051000J3")]:
    print(a, "vs", b, ": map totals", round(scr[a].sum()), round(scr[b].sum()),
          "; pixel correlation of the screen maps", round(np.corrcoef(scr[a].ravel(), scr[b].ravel())[0, 1], 3))

The two `ABS` files are recordings with a Corners-type stimulus (`CORNERS`; triangles 45 DU
wide and 60 DU deep, as the Methods give for Corners), filed under `Star`, and the spreadsheet
marks both neurons as tested with the Star. Their maps are not copies of the same neurons'
Corners files (pixel correlation 0.72 and 0.83, different totals), so they are second Corners
recordings, not Star recordings. They hold no `Standard` frame and no polygons and are not used.
The Star folder thus holds 27 Star recordings.

In [ ]:
# Recordings deposited twice: identical map totals or near-identical maps between any two files
tot = {}
for r in files.to_dict("records"):
    m = v.load_record(r, with_screen=True)["screen_smoothed"]
    tot[r["code"]] = (round(float(m.sum()), 6), m)
codes = sorted(tot)
same_total = [(a, b) for i, a in enumerate(codes) for b in codes[i + 1:] if tot[a][0] == tot[b][0]]
high = []
for i, a in enumerate(codes):
    for b in codes[i + 1:]:
        c = np.corrcoef(tot[a][1].ravel(), tot[b][1].ravel())[0, 1]
        if c > 0.9:
            high.append((a, b, round(c, 3)))
print("pairs with identical totals:", same_total)
print("pairs of maps correlated above 0.9:", high)
del tot

No recording is deposited twice. The same neuron appears in several folders under one cell code
(date, monkey letter, cell number; the trailing letter differs between conditions, as in
`ABC07240J1D`, `ABD07240J1B`, `AEC07240J1C`). The spreadsheet lists cells by this code without the
letter, so a cell code is taken to identify one neuron across conditions; nothing else in the files
confirms it, and the claims that compare conditions treat the groups as independent, as the paper
does.

In [ ]:
# Units: the one file with an eye time map
r = [x for x in recs if "eyetime" in x][0]
counts = r["screen_image"] / v.PER_MS_TO_PER_S * r["eyetime"]
print(r["code"], ": eyetime range", r["eyetime"].min(), "to", r["eyetime"].max(),
      "; image x eyetime deviates from an integer by at most", np.abs(counts - np.round(counts)).max())
print("  spikes in the map:", int(np.round(counts).sum()), "; eye time on the map:", round(r["eyetime"].sum() / 1000, 1), "s",
      "; overall rate", round(np.round(counts).sum() / (r["eyetime"].sum() / 1000), 2), "spikes/s")
print("  image (as stored) max", r["screen_image"].max() / 1000, "= one spike in one sample")
img_stored = r["screen_image"] / v.PER_MS_TO_PER_S
seen = r["eyetime"] > 0
print("  pixels with eye time:", int(seen.sum()), "; of these, image is not an integer at",
      int((np.abs(img_stored[seen] - np.round(img_stored[seen])) > 1e-9).sum()),
      f"({(np.abs(img_stored[seen] - np.round(img_stored[seen])) > 1e-9).mean():.1%}); values strictly between 0 and 1 at",
      int(((img_stored[seen] > 0) & (img_stored[seen] < 1)).sum()))
# Smoothing (Fig 3B legend: Gaussian of size 16 DU and STD 7 DU, applied after dividing by eye time)
for rr in [x for x in files.to_dict("records") if x["code"] in ("ABC050900J1A", "AEC052900J1", "ABD071200J1")]:
    q = v.load_record(rr, with_screen=True)
    x = np.arange(16) - 7.5
    g = np.exp(-x ** 2 / (2 * 7 ** 2)); k = np.outer(g, g); k /= k.sum()
    sm = ndimage.convolve(q["screen_image"], k, mode="constant")
    print(rr["code"], ": smoothed_image against image smoothed by a 16 DU, STD 7 Gaussian, r =",
          round(np.corrcoef(sm.ravel(), q["screen_smoothed"].ravel())[0, 1], 4),
          ", ratio of totals", round(q["screen_smoothed"].sum() / sm.sum(), 4))

`image` is not a spike count: multiplied by the eye time (in 1 ms samples) it gives an integer
count at every pixel, and at 98.5% of the pixels with eye time its value lies strictly between 0
and 1, so it is the unsmoothed spike rate in spikes per sample, and its maximum of 1 is one spike
in one sample. Rates are multiplied by 1000 to give spikes/s. `smoothed_image` matches `image`
smoothed by the Fig 3B Gaussian. Only one file of 77 holds the eye time, and none holds the spike
count map, so the number of spikes per condition cannot be computed (claim spikes-per-condition).


In [ ]:
# The Standard frame: the screen map moved by an offset and rotated as the stimulus polygons are
reg = []
for rr in files.to_dict("records"):
    q = v.load_record(rr, with_screen=True)
    if not q["has_standard"]:
        continue
    s, R, t, err, k0 = q["transform"]
    tries = [(s, R, t, err)]
    if q["folder"] == "Star":  # the star is symmetric under 30 degree turns: try every one
        a = q["screen_poly"]
        tries = [v.fit_rigid(np.roll(a, k, axis=1).reshape(-1, 2), q["poly"].reshape(-1, 2)) for k in range(24)]
        tries = [x for x in tries if np.linalg.det(x[1]) > 0]
    best = max(((v.register_shift(q["screen_smoothed"], *x[:3], q["smoothed"]), x) for x in tries), key=lambda y: y[0][1])
    (shift, corr), x = best
    off = x[1].T @ np.array(shift)
    reg.append({"code": q["code"], "folder": q["folder"], "polygon_fit_residual_DU": x[3], "map_corr": corr,
                "rotation_deg": np.degrees(np.arctan2(x[1][1, 0], x[1][0, 0])), "rf_offset_x": off[0], "rf_offset_y": off[1]})
reg = pd.DataFrame(reg)
print(reg.groupby("folder").agg(n=("code", "size"), max_residual=("polygon_fit_residual_DU", "max"),
                               min_corr=("map_corr", "min"), median_corr=("map_corr", "median"),
                               failed=("map_corr", lambda x: int(x.isna().sum()))).to_string())
print(reg.round({"map_corr": 4, "rotation_deg": 1, "rf_offset_x": 1, "rf_offset_y": 1}).drop(columns="polygon_fit_residual_DU").to_string(index=False))
both = reg.assign(cell=reg.code.map(lambda c: v.parse_code(c)["cell"])).groupby("cell").filter(lambda g: len(g) > 1)
print("offsets of neurons with more than one aligned condition:")
print(both.sort_values("cell")[["cell", "code", "rf_offset_x", "rf_offset_y"]].round(1).to_string(index=False))

For every file with a `Standard` struct, the stimulus polygons of the screen and of `Standard` are
the same up to a rotation and a translation, and the screen `smoothed_image`, rotated the same way
and shifted, matches the `Standard` map (correlations above; all 60 files register, none fails). So `Standard` is the authors' map in
the ROI frame of Fig 5, with the receptive-field offset applied. That offset is in no file: it
differs between conditions of the same neuron by up to tens of DU (table above), so it cannot be
carried from one condition to another, and the 15 Star files without `Standard` (and the two
`ABS` files) cannot be placed on their stimulus. All analyses use the `Standard` maps: 26 Corners,
12 Star and 22 Square and Edge recordings.

In [ ]:
aligned = [r for r in recs if r["has_standard"]]
print("aligned recordings by condition and monkey:")
print(pd.crosstab(pd.Series([r["folder"] for r in aligned], name="folder"), pd.Series([r["monkey"] for r in aligned], name="monkey")).to_string())
print("Star recordings without a Standard frame:", [r["code"] for r in recs if r["folder"] == "Star" and not r["has_standard"]])
# Geometry: luminance along every fold at 5%, 50% and 95% of its length, and in the baseline squares
geo = []
for r in aligned:
    F = v.folds(r, "both" if r["folder"] == "Corners" else "bright")
    row = {"code": r["code"], "folder": r["folder"], "frame": "x".join(map(str, r["smoothed"].shape))}
    for k, segs in F.items():
        row[k + " length"] = round(float(np.mean([np.linalg.norm(b - a) for a, b in segs])), 1)
        for f in (0.05, 0.5, 0.95):
            row[f"{k} lum{int(f * 100)}"] = round(v.lum_at_fraction(r, segs, f), 1)
    if r["folder"] == "Corners":
        for place in v.CORNERS_PLACES:
            row[f"baseline lum max, {place}"] = round(max(v.box_mean(r["lum"], c, (1, 0), 16, 16)
                                                          for c in v.corners_baseline_centres(r, place)), 2)
    else:
        row["baseline lum max, inside"] = round(max(v.box_mean(r["lum"], c, (1, 0), 16, 16) for c in v.baseline_centres_inside(r)), 2)
        if r["folder"] == "Star":
            row["baseline lum max, centred"] = round(max(v.box_mean(r["lum"], c, (1, 0), 16, 16) for c in v.baseline_centres(r)), 2)
    if r["folder"] == "Corners":
        row["bright/dark S40 folds"] = r["corners_polarity"]
    geo.append(row)
geo = pd.DataFrame(geo)
for f, g in geo.groupby("folder"):
    print(f)
    print(g.dropna(axis=1, how="all").drop(columns=["code", "folder"]).describe().loc[["count", "min", "max"]].round(1).to_string())
print("Corners: bright and dark S40 folds per file:", geo["bright/dark S40 folds"].dropna().value_counts().to_dict())
# Square and Edge: the centred baseline squares run past the authors' 151 DU frame, so they are read
# from the screen maps moved into the frame widened by v.PAD DU (vasarely.load_record)
sq = [r for r in aligned if r["folder"] == "Edge-compare"]
inner = [np.corrcoef(r["padded"]["smoothed"][v.PAD:-v.PAD, v.PAD:-v.PAD].ravel(), r["smoothed"].ravel())[0, 1] for r in sq]
w_ = sq[0]["smoothed"].shape[1]
outside = [sum((c[0] - 8 < 0.5) or (c[0] + 8 > w_ + 0.5) for c in v.baseline_centres(r)) for r in sq]
print("Square and Edge, widened map against the authors' map inside their frame: r from", round(min(inner), 4), "to", round(max(inner), 4))
off = np.array([[1 - v.box_mean(r["padded"]["on_screen"], c + v.PAD, (1, 0), 16, 16) for c in v.baseline_centres(r)] for r in sq])
print("  centred baseline squares partly outside the authors' frame:", sum(outside), "of", 4 * len(sq),
      "; squares partly off the screen:", int((off > 0.01).sum()), "in", int((off > 0.01).any(1).sum()), "files, at most",
      round(off.max(), 2), "of a square; the baseline averages the part on the screen")
print("  stimulus luminance in the centred squares on the widened frame: at most",
      round(max(v.box_mean(r["padded"]["lum"], c + v.PAD, (1, 0), 16, 16) for r in sq for c in v.baseline_centres(r)), 2), "%")
print("Edge files whose Edge is cut by the screen border (luminance at 95% below 90):",
      geo.loc[geo["Edge lum95"] < 90, "code"].tolist() if "Edge lum95" in geo else [])

Along each fold the stimulus runs from 5% to 95% luminance at 5% and 95% of the fold's length and
is 50% at its middle, so the 50% midpoints and the ten blocks are placed as the Methods describe.
Fold lengths give the block lengths of the Results (15 DU for Edge, P180 and S52, 7.07 for S90,
10 for S40). The Corners pattern has two S40 folds, one bright and one dark, in every file. The
Corners luminance is 1% at its darkest (the image holds no 0); every Corners baseline square lies
there, and every Square and Star baseline square on the 0% background. The authors' Square and
Edge frame is 151 DU wide, so the 16 x 16 DU squares centred on the corners of the 141.4 DU
circumscribing square run past it on both sides (all 88 squares). They are read from the screen
maps moved into a frame widened by 16 DU, which match the authors' maps inside their frame (r of
0.994 or more) and are 0% there too; 6 squares in 5 files lie partly off the screen, and their
baseline is the mean over the part on the screen. In four Square and Edge files the 100% end of
the Edge is cut by the screen border, which the Methods say happened to the rectangle "but never
its edge"; the Edge measures there partly sample the black screen beyond the border. The paper's
ROI for the Star is 10 x 10 DVA (160 DU), but the authors' Star frame is 321 DU wide, which the
300 DU star needs.


In [ ]:
# Does the response map line up with the stimulus? The maps have no clock; their counterpart of the
# trial table is the stimulus. The Edge is a real luminance edge aligned with each neuron's preferred
# orientation, so an aligned map should peak on it. Profile of the spike rate across the Edge (8 DU
# wide strips parallel to it, over its 25% to 75% stretch), the map against itself moved sideways.
offsets = np.arange(-48, 49, 8)
prof = []
for r in aligned:
    if r["folder"] != "Edge-compare":
        continue
    p0, p100 = v.folds(r)["Edge"][0]
    d = p100 - p0; n = np.array([-d[1], d[0]]) / np.linalg.norm(d)
    into = r["rect"][0].mean(0) - (p0 + p100) / 2
    n = n if np.dot(n, into) > 0 else -n
    prof.append([v.box_mean(r["smoothed"], (p0 + p100) / 2 + o * n, d, 75, 8) for o in offsets])
prof = np.array(prof)
peak = offsets[np.argmax(prof, 1)]
print("offset across the Edge (DU; + is into the rectangle):", offsets.tolist())
print("mean rate (spikes/s):                               ", np.round(prof.mean(0), 1).tolist())
print("neurons whose profile peaks within 8 DU of the Edge:", int((np.abs(peak) <= 8).sum()), "of", len(peak),
      "; peak offsets:", sorted(peak.tolist()))
print("on the Edge against 48 DU either side, per neuron: higher in", int(((prof[:, 6] > prof[:, 0]) & (prof[:, 6] > prof[:, -1])).sum()), "of", len(prof))

The maps have no time base, so the check of a shared clock becomes a check that each map sits on
its stimulus. Across the Edge, which runs along each neuron's preferred orientation, the mean rate
peaks on the Edge (26.6 spikes/s, against 10.1 at 48 DU outside and 12.4 at 48 DU inside the
rectangle); 16 of 22 neurons peak within 8 DU of it and 19 of 22 are higher on it than at 48 DU on
either side. This is consistent with the authors' alignment of the Square and Edge files; it does
not show it for the six neurons that peak elsewhere. No check of the same kind is independent of
the claims for Star and Corners, whose only features along the preferred orientation are the folds
under test; their alignment is the authors' and is taken as it is.

In [ ]:
def summarize(df, col="diff"):
    g = df.groupby("fold")[col]
    out = pd.DataFrame({"n": g.size(), "mean": g.mean(), "sem": g.sem(), "median": g.median(), "min": g.min(), "max": g.max()})
    return out.reindex([f for f in v.FOLD_ORDER if f in out.index])

PAPER_MID = {"S40": (25.60, 4.18), "S52": (18.10, 2.39), "S90": (10.69, 1.94), "P180": (4.87, 1.15), "Edge": (22.01, 3.41)}
main = v.neuron_table(aligned, corners_mode="bright", baseline_reading="matched")
both = v.neuron_table(aligned, corners_mode="both", baseline_reading="matched")
# every baseline reading, with the bright S40 fold: (name, table)
READINGS = {"matched [main]": main,
            "centred": v.neuron_table(aligned, "bright", "centred"),
            "inside": v.neuron_table(aligned, "bright", "inside"),
            "files' BaseActivity": v.neuron_table(aligned, "bright", "files").dropna(subset=["base"]),
            **{f"matched, Corners {p}": v.neuron_table(aligned, "bright", "matched", corners_place=p) for p in v.CORNERS_PLACES[1:]}}
has_files = set(READINGS["files' BaseActivity"].code)
READINGS["matched, files with BaseActivity only"] = main[main.code.isin(has_files)]
print("variables used (spikes/s), main reading:")
print(summarize(main, "mid").round(2).to_string())
print(summarize(main, "base").round(2).to_string())
print("computed baselines against the files' BaseActivity, per file (spikes/s):")
cmp = []
for r in aligned:
    if not np.isfinite(r["base_activity"]):
        continue
    row = {"code": r["code"], "folder": r["folder"], "files": r["base_activity"]}
    if r["folder"] == "Corners":
        for place in v.CORNERS_PLACES:
            row[place] = v.baseline_value(r, "smoothed", "centred", place)
    else:
        row["centred"] = v.baseline_value(r, "smoothed", "centred")
        row["inside"] = v.baseline_value(r, "smoothed", "inside")
        row["centred, unsmoothed"] = v.baseline_value(r, "raw", "centred")
        row["inside, unsmoothed"] = v.baseline_value(r, "raw", "inside")
    cmp.append(row)
cmp = pd.DataFrame(cmp)
print(cmp.round(2).to_string(index=False))
print("median |computed - files| and Spearman r with the files' values, by stimulus:")
for f, g in cmp.groupby("folder"):
    cols = [c for c in g.columns if c not in ("code", "folder", "files") and g[c].notna().all()]
    print(f"  {f} (n = {len(g)}): " + "; ".join(f"{c} {np.median(np.abs(g[c] - g.files)):.2f}, r = {stats.spearmanr(g[c], g.files)[0]:.2f}" for c in cols))
# Squares moved inward step by step: which offset comes closest to BaseActivity
print("median |computed - files| with the squares moved k DU inward from centred (k = 8 is 'inside'):")
for f in ["Edge-compare", "Star"]:
    rs = [r for r in aligned if r["folder"] == f and np.isfinite(r["base_activity"])]
    line = []
    for k in range(0, 17, 2):
        e = []
        for r in rs:
            c0 = r["poly"][-1].mean(0); half = np.abs(r["poly"][0] - c0).max() - k
            cs = [c0 + np.array([sx, sy]) * half for sx in (-1, 1) for sy in (-1, 1)]
            if f == "Edge-compare":  # on the widened map, over the part of the squares on the screen
                b = v.baseline(r["padded"]["smoothed"], [c + v.PAD for c in cs]) / v.baseline(r["padded"]["on_screen"], [c + v.PAD for c in cs])
            else:
                b = v.baseline(r["smoothed"], cs)
            e.append(abs(b - r["base_activity"]))
        line.append(f"k={k}: {np.median(e):.2f}")
    print(f"  {f}: " + ", ".join(line))
fig, ax = plt.subplots(1, 3, figsize=(13, 3.8))
for a, f in zip(ax, ["Edge-compare", "Star", "Corners"]):
    g = cmp[cmp.folder == f]
    for c, mk in zip([c for c in g.columns if c not in ("code", "folder", "files") and "unsmoothed" not in c and g[c].notna().all()], "osd"):
        a.plot(g.files, g[c], mk, label=c, alpha=0.7)
    hi = float(np.nanmax(g.drop(columns=["code", "folder"]).values)) * 1.05
    a.plot([0, hi], [0, hi], ":k"); a.set_title(f"{f} (n = {len(g)})"); a.set_xlabel("files' BaseActivity (spikes/s)"); a.legend(fontsize=7)
ax[0].set_ylabel("computed baseline (spikes/s)"); plt.tight_layout(); plt.show()
print("all points are drawn (axes from 0 to the largest value)")


The text does not fix where the baseline squares are: centred on the corners of the circumscribing
square, or drawn inside them, and for Corners where in the 0% area. The files' own `BaseActivity`,
held by 38 of 60 aligned files, is the authors' value, so the placements are compared with it file
by file above. For Square and Edge the squares centred on the corners come closest (median
difference 0.46 spikes/s, against 1.33 for squares inside the corners, n = 9), and moving them
inward only moves them away. For the Star the squares inside the corners come closest (0.39,
against 1.30 for centred, n = 10; Spearman r 0.98 against 0.58). For Corners none of the three
placements tried matches closely (0.85 to 2.79). The main reading, called "matched", takes for each
stimulus the placement closest to `BaseActivity`: centred for Square and Edge, inside for the
Star, the bottom corners of the ROI for Corners. This is an assumption. The centred and inside
readings for both stimuli, the files' values themselves, two other Corners placements and the
main reading on the 38 files with `BaseActivity` are shown with every claim the baseline bears on.

## neuron-counts

> Of 118 V1 neurons recorded from two monkeys (102 from monkey J, 16 from monkey A), 60 were
> discarded for technical problems and 58 analysed; 7 were tested in all three conditions, 11 in
> two and 40 in one, giving 25 neurons for Square and Edge, 30 for Star and 28 for Corners.

In [ ]:
def counts(cond_sets):
    k = pd.Series({c: len(s) for c, s in cond_sets.items()})
    per = {cond: sum(cond in s for s in cond_sets.values()) for cond in ["Square and Edge", "Star", "Corners"]}
    return {"neurons": len(cond_sets), **{f"in {c}": n for c, n in per.items()},
            "three": int((k == 3).sum()), "two": int((k == 2).sum()), "one": int((k == 1).sum())}

name = {"Corners": "Corners", "Star": "Star", "Edge-compare": "Square and Edge"}
by_files = {}
for r in recs:
    by_files.setdefault(r["cell"], set()).add(v.PREFIX_CONDITION[r["prefix"]] if r["prefix"] != "ABD" else "Star")
by_folder = {}
for r in recs:
    by_folder.setdefault(r["cell"], set()).add(name[r["folder"]])
by_sheet = {c: {name[folder_of[k]] for k, m in row.items() if m} for c, row in zip(sheet["cell"], marks.to_dict("records"))}
by_aligned = {}
for r in aligned:
    by_aligned.setdefault(r["cell"], set()).add(name[r["folder"]])
paper = {"neurons": 58, "in Square and Edge": 25, "in Star": 30, "in Corners": 28, "three": 7, "two": 11, "one": 40}
tab = pd.DataFrame({"paper": paper, "spreadsheet (all rows)": counts(by_sheet),
                    "files, by folder": counts(by_folder), "files, by stimulus type": counts(by_files),
                    "aligned files": counts(by_aligned)})
print(tab.to_string())
print("monkeys among neurons with files:", pd.Series([c[-2] for c in by_files]).value_counts().to_dict(),
      "; paper: 102 recorded from J and 16 from A, before 60 were discarded")

**Result: not reproduced.** The spreadsheet has 58 rows, the paper's total, but 5 of them mark no
condition and have no files, so the files hold 53 neurons (49 from monkey J, 4 from A), as the
data descriptor says. By folder they give 26 Corners, 29 Star and 22 Square and Edge neurons; by
stimulus type (the two `ABS` files are Corners recordings) 26, 27 and 22. None of the three
per-condition counts is the paper's (28, 30, 25). Of the overlap counts, 7 neurons in all three
conditions matches the paper, but 10 (or 8 by stimulus type) in two and 36 (or 38) in one do not
(11 and 40). The analysable files, those with the authors' alignment, cover 43 neurons. The
recorded totals per monkey (102 and 16) cannot be checked, as the discarded neurons are not in the
data set.

## spikes-per-condition

> Each experimental condition required at least 10,000 spikes, and the mean number of spikes per
> condition was 35,998 (STD 25,981).

Blocked: the files hold no spike count map and, but for one file, no eye time map (data check),
so the number of spikes cannot be computed under either reading of the criterion. In the one file
with an eye time map, `ABC050900J1A`, the map holds 15,644 spikes (printed in the data check).

## s4-baseline-no-difference

> Absolute baseline activity did not differ significantly across the three visual stimuli
> (Square and Edge, Star, Corners).

In [ ]:
def baseline_tests(vals):
    groups = [vals[c] for c in ["Square and Edge", "Star", "Corners"]]
    pairs = [("Square and Edge", "Star"), ("Square and Edge", "Corners"), ("Star", "Corners")]
    p_pair = [stats.ranksums(vals[a], vals[b]).pvalue for a, b in pairs]
    p_bh = multipletests(p_pair, method="fdr_bh")[1]
    out = {f"(a) {a} vs {b}, BH p": p for (a, b), p in zip(pairs, p_bh)}
    out["(b) Kruskal-Wallis p"] = stats.kruskal(*groups).pvalue
    out["(c) one-way ANOVA p"] = stats.f_oneway(*groups).pvalue
    return out

s4 = {}
for reading, df in READINGS.items():
    one = df.drop_duplicates("code")  # one baseline per recording
    vals = {c: one.loc[one.condition == c, "base"].values for c in ["Square and Edge", "Star", "Corners"]}
    print(f"baseline reading: {reading}")
    for c in ["Square and Edge", "Star", "Corners"]:
        x = np.array(vals[c])
        print(f"  {c:16s} n = {len(x):2d}, median {np.median(x):6.2f}, quartiles {np.percentile(x, 25):6.2f} to {np.percentile(x, 75):6.2f}, "
              f"range {x.min():.2f} to {x.max():.2f} spikes/s")
    s4[reading] = baseline_tests(vals)
    for k, p in s4[reading].items():
        print(f"  {k}: {p:.3f}")
print("readings finding a difference at 0.05 (any of a, b, c):", [k for k, t in s4.items() if min(t.values()) < 0.05])
print("readings finding none:", [k for k, t in s4.items() if min(t.values()) >= 0.05])

**Result: not reproduced.** Under the main reading all three tests find a difference at alpha
0.05: (a) Square and Edge against Corners, BH p = 0.037 (Square and Edge against Star 0.065), (b)
Kruskal-Wallis p = 0.025, (c) ANOVA p = 0.004. The Square and Edge baseline is the highest (median
6.72 spikes/s, against 1.90 for Star and 2.57 for Corners). The squares centred or inside the
corners for both stimuli find a larger difference (Kruskal-Wallis p = 0.005 and 0.008). With the
files' own `BaseActivity` no test finds a difference (smallest p = 0.109), and neither does the main
reading on the same 38 files (smallest p = 0.060) or with the Corners squares 32 DU further in
(smallest p = 0.063). The outcome rests on where the baseline squares are, which the text does not
fix, and on fewer neurons per stimulus than the paper's (22, 12 and 26 against 25, 30 and 28).

## fig4a-above-baseline

> For every fold (P180, S90, S52, S40), most neurons' midpoint fold responses lie above their
> absolute baseline activity, and the mass center of each fold's points lies above the line of
> unit slope, with the mass center's distance from that line increasing as the fold angle
> decreases.

In [ ]:
folds4 = ["P180", "S90", "S52", "S40"]
def fig4a(df):
    rows = []
    for f in folds4:
        d = df[df.fold == f]
        pos = (d.base > 0) & (d.mid > 0)
        rows.append({"fold": f, "n": len(d), "above": int((d.mid > d.base).sum()), "fraction above": (d.mid > d.base).mean(),
                     "(a) centre base": d.base.mean(), "(a) centre resp": d.mid.mean(), "(a) distance": d.mid.mean() - d.base.mean(),
                     "(b) n positive": int(pos.sum()),
                     "(b) distance log10": np.log10(d.mid[pos]).mean() - np.log10(d.base[pos]).mean()})
    return pd.DataFrame(rows).set_index("fold")
for label, df in READINGS.items():
    t = fig4a(df)
    print(label)
    print(t.round(3).to_string())
    for col in ["(a) distance", "(b) distance log10"]:
        print(f"  ordered P180 < S90 < S52 < S40 by {col}:", bool(np.all(np.diff(t[col].values) > 0)))
fig, ax = plt.subplots(1, 4, figsize=(14, 3.5))
for a, f in zip(ax, folds4):
    d = main[main.fold == f]
    a.loglog(d.base.clip(lower=0.05), d.mid.clip(lower=0.05), "o", ms=4)
    a.plot([0.05, 200], [0.05, 200], ":k")
    a.plot(d.base.mean(), d.mid.mean(), "r+", ms=12)
    a.set_title(f"{f} (n = {len(d)})"); a.set_xlabel("baseline (spikes/s)")
ax[0].set_ylabel("midpoint response (spikes/s)")
plt.tight_layout(); plt.show()
print("values below 0.05 spikes/s drawn at 0.05:", int(((main[main.fold.isin(folds4)][["base", "mid"]] < 0.05).sum()).sum()))

**Result: reproduced.** Under the main reading most neurons respond above baseline for each fold
(18 of 22 for P180, 18 of 22 for S90, 12 of 12 for S52, 26 of 26 for S40), every mass centre lies
above the unit line, and its distance from the line grows from P180 through S90 and S52 to S40,
under reading (a) (4.03, 8.39, 19.19, 26.27 spikes/s) and under reading (b) (log ratios 0.221,
0.351, 0.916, 1.002). The order under (a) holds under every baseline reading shown; under (b) it
fails with the Star squares centred on the corners (S52 1.210 > S40 1.002) and with the Corners
squares 32 DU further in (S40 0.837 < S52 0.916).

## fig4b-spike-rate-difference

> The mean spike rate difference at the fold midpoint decreases with increasing fold angle, from
> S40 (25.60 spikes/s) through S52 (18.10) and S90 (10.69) to P180 (4.87).

In [ ]:
def compare(df, label):
    s = summarize(df).reindex(folds4)
    s["paper"] = [PAPER_MID[f][0] for f in folds4]
    s["tolerance"] = [PAPER_MID[f][1] for f in folds4]
    s["within"] = (s["mean"] - s["paper"]).abs() <= s["tolerance"]
    order = bool(np.all(np.diff(s["mean"].values) < 0) is False) if False else bool(all(s["mean"].values[i] < s["mean"].values[i + 1] for i in range(3)))
    print(label, "| order S40 > S52 > S90 > P180:", order, "| all within tolerance:", bool(s["within"].all()))
    print(s.round(2).to_string())
    return s
fig4b_main = compare(main, "bright S40 fold [the claims file's midpoint]")
compare(both, "both S40 folds averaged (reading (a) before the claims file was amended; reference only)")
compare(v.neuron_table(aligned, "dark", "matched"), "dark S40 fold alone (reference)")
fig4b_var = {}
for label, df in list(READINGS.items())[1:] + [("matched, unsmoothed map", v.neuron_table(aligned, "bright", "matched", map_name="raw"))]:
    fig4b_var[label] = compare(df, f"bright S40, baseline {label}")
print("readings with the order and all four within tolerance:",
      [k for k, s in fig4b_var.items() if s["within"].all() and all(s["mean"].values[i] < s["mean"].values[i + 1] for i in range(3))])
print("by monkey (main reading):")
print(main[main.fold.isin(folds4)].groupby(["fold", "monkey"])["diff"].agg(["size", "mean"]).round(2).unstack().reindex(folds4).to_string())

**Result: partially reproduced.** Under the main reading the means are ordered S40 (26.27) > S52
(19.19) > S90 (8.39) > P180 (4.03 spikes/s); S40, S52 and P180 are within the paper's SEM, but S90
is 2.30 below 10.69, outside its 1.94. The order holds under every baseline reading shown and on
the unsmoothed map, and under none of them are all four within tolerance: S90 falls outside with
the squares centred or inside for both stimuli and with both other Corners placements; P180 and
S52 with the files' `BaseActivity`; S52 and S40 with the main reading on the files that have it;
P180 on the unsmoothed map. The two monkey A neurons give negative P180 and S90 values (-2.94 and
-4.44) against 4.72 and 9.68 for monkey J. The Star mean comes from 12 neurons, not 30.

## fig4b-pairwise-significance

> The midpoint spike rate difference for S40 is significantly larger than for S90 and for P180, and
> that for S52 is significantly larger than for P180.

In [ ]:
def pairwise(df, label):
    vals = {f: df.loc[df.fold == f, "diff"].values for f in folds4}
    all6 = [(a, b) for i, a in enumerate(["S40", "S52", "S90", "P180"]) for b in ["S40", "S52", "S90", "P180"][i + 1:]]
    named = [("S40", "S90"), ("S40", "P180"), ("S52", "P180")]
    raw = {pr: stats.ranksums(vals[pr[0]], vals[pr[1]]) for pr in all6}
    out = pd.DataFrame({"z": {pr: raw[pr].statistic for pr in all6}, "p": {pr: raw[pr].pvalue for pr in all6},
                        "median difference": {pr: np.median(vals[pr[0]]) - np.median(vals[pr[1]]) for pr in all6}})
    out["(a) BH over 6"] = multipletests(out.p.values, method="fdr_bh")[1]
    out["(b) BH over 3"] = np.nan
    out.loc[named, "(b) BH over 3"] = multipletests(out.loc[named, "p"].values, method="fdr_bh")[1]
    print(label)
    print(out.to_string(float_format=lambda x: f"{x:.4g}"))
    for col in ["(a) BH over 6", "(b) BH over 3"]:
        ok = all(out.loc[pr, col] < 0.05 and out.loc[pr, "z"] > 0 for pr in named)
        print(f"  the three named pairs significant and in the stated direction under {col}:", ok)
pairwise(main, "bright S40 fold [main]")
for label, df in list(READINGS.items())[1:]:
    pairwise(df, f"bright S40, baseline {label}")

**Result: reproduced.** Under the main reading, S40 against S90 (BH p = 0.00028 over six pairs)
and against P180 (p = 2.1e-06), and S52 against P180 (p = 0.00014), are significant in the stated
direction under both comparison families, and under every other baseline reading shown. Of the
pairs the text does not claim, S52 against S90 is significant (p = 0.011), and S40 against S52 (p =
0.36) and S90 against P180 (p = 0.16) are not.

## fig5-average-maps

> The population-average spike rate difference maps show neural activity increasing as the corner
> angle decreases.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 6))
avg = {}
for a, folder in zip(ax, ["Edge-compare", "Star", "Corners"]):
    rs = [r for r in aligned if r["folder"] == folder]
    shapes = pd.Series(["x".join(map(str, r["smoothed"].shape)) for r in rs]).value_counts()
    print(folder, "frame sizes (rows x columns, DU) and files:", dict(shapes))
    # frames of different sizes are padded about their centre onto the largest
    H = max(r["smoothed"].shape[0] for r in rs); W = max(r["smoothed"].shape[1] for r in rs)
    stack = []
    for r in rs:
        m = r["smoothed"] - v.baseline_value(r)
        dy, dx = (H - m.shape[0]) // 2, (W - m.shape[1]) // 2
        canvas = np.full((H, W), np.nan); canvas[dy:dy + m.shape[0], dx:dx + m.shape[1]] = m
        stack.append(canvas)
    mean_map = np.nanmean(np.array(stack), 0)
    avg[folder] = (mean_map, rs[[r["smoothed"].shape for r in rs].index((H, W))])
    im = a.imshow(mean_map, cmap="hot", vmin=-5, vmax=40)
    a.contour(avg[folder][1]["lum"], levels=[1.5, 50, 98.5], colors=["c", "g", "b"], linewidths=0.8)
    a.set_title(f"{folder} (n = {len(rs)}; frames {dict(shapes)})", fontsize=9)
    print(folder, "average map range", np.nanmin(mean_map).round(2), "to", np.nanmax(mean_map).round(2), "spikes/s (colour scale -5 to 40)")
plt.colorbar(im, ax=ax, shrink=0.6, label="spike rate difference (spikes/s)"); plt.show()
rows = []
for folder, (m, ref) in avg.items():
    for k, segs in v.folds(ref, "bright").items():
        if k == "Edge":
            continue
        rows.append({"fold": k, "midpoint of the average map": v.midpoint_response(m, segs),
                     "mean of its ten blocks": v.block_profile(m, segs).mean()})
f5 = pd.DataFrame(rows).set_index("fold")
print(f5.round(2).to_string())
mid = f5["midpoint of the average map"]
print("order S40 > S52 > S90 > P180 (bright S40):", bool(mid["S40"] > mid["S52"] > mid["S90"] > mid["P180"]))
blk = f5["mean of its ten blocks"]
print("variant, block means, order (bright S40):", bool(blk["S40"] > blk["S52"] > blk["S90"] > blk["P180"]))

**Result: reproduced.** At the fold midpoints the average maps are ordered S40 (26.27) > S52
(19.19) > S90 (8.39) > P180 (4.03 spikes/s), and the variant of block means gives the same order
(23.01, 17.15, 7.61, 3.61). As the claims file says, this repeats the means of
fig4b-spike-rate-difference and is not an independent test.

## Table 1: table1-edge, table1-p180, table1-s90, table1-s52, table1-s40

> Along each fold or the Edge, the minimum, maximum, mean and median of the spike rate difference
> over the ten 10% luminance blocks, and the midpoint value (Table 1).

In [ ]:
TABLE1 = {"Edge": [(7.98, 1.60), (33.13, 5.11), (20.44, 3.33), (20.77, 3.48), (22.01, 3.41)],
          "P180": [(0.18, 1.01), (10.28, 1.53), (4.55, 1.12), (4.03, 1.06), (4.87, 1.15)],
          "S90": [(4.36, 1.07), (14.61, 2.48), (9.22, 1.61), (9.31, 1.62), (10.69, 1.94)],
          "S52": [(6.26, 1.33), (25.39, 3.52), (15.66, 2.23), (15.93, 2.36), (18.10, 2.39)],
          "S40": [(9.33, 1.69), (35.11, 4.17), (22.81, 3.11), (22.83, 3.48), (25.60, 4.18)]}
STATS = ["min", "max", "mean", "median", "midpoint"]

def table1(df, fold, halves, order):
    d = df[df.fold == fold]
    if order == "a":  # per neuron, then the mean over neurons
        s = v.block_summaries(d, halves)
        vals = list(s.mean(0)) + [d["diff"].mean()]
        sems = list(s.std(0, ddof=1) / np.sqrt(len(s))) + [d["diff"].sem()]
    else:  # summaries of the population-average profile
        prof = np.array([b.mean(0) if halves == "averaged" else b for b in d.blocks])
        if halves == "averaged":
            p = prof.mean(0); s = [p.min(), p.max(), p.mean(), np.median(p)]
        else:
            p = prof.mean(0); s = list(np.mean([[x.min(), x.max(), x.mean(), np.median(x)] for x in p], 0))
        vals = s + [d["diff"].mean()]; sems = [np.nan] * 5
    return vals, sems

# S40: the amended claims file takes the bright fold alone; the two folds averaged (its former
# reading (i)) are printed for reference only
t1 = {}
for fold in ["Edge", "P180", "S90", "S52", "S40"]:
    variants = [("i", "a"), ("i", "b")] + ([("ii", "a"), ("ii", "b")] if fold in ("S90", "S52", "S40") else [])
    for halves_label, order in variants:
        for mode in (["bright", "both"] if fold == "S40" else ["bright"]):
            df = main if mode == "bright" else both
            halves = "averaged" if halves_label == "i" else "separate"
            if fold == "S40" and mode == "bright" and halves_label == "ii":
                continue  # one fold: the two readings coincide
            vals, sems = table1(df, fold, halves, order)
            within = [abs(x - e[0]) <= e[1] for x, e in zip(vals, TABLE1[fold])]
            label = f"{fold} ({halves_label}{order})" + (" bright fold" if fold == "S40" and mode == "bright" else " both folds (ref.)" if fold == "S40" else "")
            t1[label] = {**{s: round(x, 2) for s, x in zip(STATS, vals)}, "SEMs": " ".join(f"{x:.2f}" for x in sems),
                         "within tolerance": sum(within)}
            print(f"{label:28s} " + "  ".join(f"{s} {x:6.2f} (paper {e[0]:5.2f}±{e[1]:.2f}) {'ok' if w else '--'}"
                                              for s, x, e, w in zip(STATS, vals, TABLE1[fold], within)))
t1 = pd.DataFrame(t1).T
print()
print("values within tolerance under reading (ia), by baseline reading (bright S40 fold):")
complete_codes = set(geo.loc[geo["Edge lum95"] >= 90, "code"])
for reading, df in list(READINGS.items()) + [("matched, the 18 files with the whole Edge on the screen",
                                              main[(main.condition != "Square and Edge") | main.code.isin(complete_codes)])]:
    best = {}
    for fold in ["Edge", "P180", "S90", "S52", "S40"]:
        n_in = {f"{h}{o}": int(sum(abs(x - e[0]) <= e[1] for x, e in zip(table1(df, fold, "averaged" if h == "i" else "separate", o)[0], TABLE1[fold])))
                for h in (["i", "ii"] if fold in ("S90", "S52") else ["i"]) for o in "ab"}
        best[fold] = f"(ia) {n_in['ia']}, best {max(n_in.values())} {[k for k, x in n_in.items() if x == max(n_in.values())]}"
    print(f"  {reading:56s}", best)
d18 = main[main.code.isin(complete_codes)]
print("Edge (ia) on the", d18.code.nunique(), "files with the whole Edge on the screen:",
      "  ".join(f"{s} {x:.2f} (paper {e[0]:.2f}±{e[1]:.2f}) {'ok' if abs(x - e[0]) <= e[1] else '--'}"
                for s, x, e in zip(STATS, table1(d18, "Edge", "averaged", "a")[0], TABLE1["Edge"])))
print("Edge on the 18 files with the whole Edge on the screen, by baseline reading (values within tolerance):")
for reading, df in READINGS.items():
    d_ = df[df.code.isin(complete_codes)]
    if d_[d_.fold == "Edge"].empty:
        continue
    n_in = {o: int(sum(abs(x - e[0]) <= e[1] for x, e in zip(table1(d_, "Edge", "averaged", o)[0], TABLE1["Edge"]))) for o in "ab"}
    vals_ = table1(d_, "Edge", "averaged", "a")[0]
    print(f"  {reading:40s} n = {d_[d_.fold == 'Edge'].code.nunique():2d}; (ia) {n_in['a']}, (ib) {n_in['b']}; (ia) values",
          " ".join(f"{x:.2f}" for x in vals_))


In [ ]:
# Profiles along each fold (population mean of the ten blocks, spike rate difference, from 0% to 100%)
fig, ax = plt.subplots(1, 5, figsize=(18, 3.2))
for a, fold in zip(ax, ["Edge", "P180", "S90", "S52", "S40"]):
    d = main[main.fold == fold]
    prof = np.array([b.mean(0) for b in d.blocks])
    m, se = prof.mean(0), prof.std(0, ddof=1) / np.sqrt(len(prof))
    a.errorbar(np.arange(5, 100, 10), m, se, fmt="o-")
    a.set_title(f"{fold} (n = {len(d)})"); a.set_xlabel("luminance (%)")
    print(fold, "block means", np.round(m, 2), "range", round(prof.min(), 2), "to", round(prof.max(), 2))
ax[0].set_ylabel("spike rate difference (spikes/s)"); plt.tight_layout(); plt.show()
print("by monkey, block mean over neurons (reading ia):")
print(main.assign(bm=[b.mean() for b in main.blocks]).groupby(["fold", "monkey"])["bm"].agg(["size", "mean"]).round(2).unstack().to_string())

**Result: by claim.** The rule is all five values within the paper's SEM for reproduced and at
least three for partially reproduced, under any one reading. The Edge values are taken from the 18
files whose Edge is wholly on the screen, as the Methods say it always was (Methods, Square and
edge: the rectangle "but never its edge" was partly off the display); this is an assumption, and
all 22 files are shown.

- table1-edge: reproduced; on the 18 files all five are within tolerance under (ia) (minimum 6.52,
  maximum 31.53, mean 18.56, median 18.76, midpoint 21.09 spikes/s). On all 22 files three are
  (the minimum 5.78 and the median 17.09 fall outside). On the 18 files all five stay within
  tolerance under (ia) with the squares inside the corners (6.57, 31.59, 18.62, 18.82, 21.15) and
  with every Corners placement, which leaves the Edge unchanged. On the 6 of the 18 files that
  hold `BaseActivity` none is within tolerance, with the files' values and with the main reading
  alike, so on the Edge the result depends on which neurons are included more than on where the
  computed squares lie.
- table1-p180: partially reproduced; four of five under (ia) (maximum 10.33, mean 3.61, median
  3.03, midpoint 4.03); the minimum, -1.55, is below 0.18 ± 1.01.
- table1-s90: not reproduced; at most two of five under any reading. Under (ia) the maximum (13.58)
  and the mean (7.61) are within tolerance; the minimum (2.46), median (7.66) and midpoint (8.39)
  are below it.
- table1-s52: reproduced; under (iia), each half of the diagonal as its own ten blocks, all five
  are within tolerance (7.38, 28.62, 17.15, 16.55, 19.19); under (ia) four are (the minimum, 9.08,
  is above 6.26 ± 1.33). On 12 neurons in place of 30.
- table1-s40: reproduced; all five under (a) (10.52, 34.26, 23.01, 23.02, 26.27).

These outcomes rest on the baseline placement and on which neurons are included (the counts by
reading printed above, best over the readings of each claim). With the squares centred for both
stimuli S52 has 3 of 5; with the squares inside for both, Edge (22 files) has 2, P180 3 and S90 1;
with the files' `BaseActivity` (9, 10 and 19 neurons) Edge has 0, P180 3, S90 3 and S52 2; with the
Corners squares 32 DU further in, S40 has 4. The main reading on the files that have
`BaseActivity` gives P180 5, S90 5, S52 1 and S40 3, and restricting P180 and S90 to the 18 files
with the whole Edge on the screen gives 2 and 1. The Star claims rest on the 12 of 27 Star
recordings that hold the authors' aligned map, a selection made by the files, not by chance.

## fig6-midpoint-representative

> For each fold and the Edge, the spike rate difference at the midpoint does not differ
> significantly from the mean or the median spike rate difference along the fold or Edge, so the
> midpoint response represents the whole fold.

In [ ]:
rows = []
for mode, df in [("bright S40 [main]", main)]:
    for fold in ["Edge", "P180", "S90", "S52", "S40"]:
        if mode != "bright S40 [main]" and fold != "S40":
            continue
        d = df[df.fold == fold]
        mids = d["diff"].values
        for halves in ["averaged", "separate"] if fold in ("S90", "S52", "S40") else ["averaged"]:
            s = v.block_summaries(d, halves)
            for j, stat in [(2, "mean"), (3, "median")]:
                rs_ = stats.ranksums(mids, s[:, j]); sr = stats.wilcoxon(mids, s[:, j])
                rows.append({"reading": mode, "fold": fold, "halves": halves, "against": stat, "n": len(d),
                             "median difference": np.median(mids - s[:, j]), "rank sum p": rs_.pvalue, "signed rank p (variant)": sr.pvalue})
f6 = pd.DataFrame(rows)
print(f6.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
m6 = f6[(f6.reading == "bright S40 [main]") & (f6.halves == "averaged")]
d18e = main[(main.fold == "Edge") & main.code.isin(complete_codes)]
s18 = v.block_summaries(d18e, "averaged")
print("Edge on the", len(d18e), "files with the whole Edge on the screen, rank sum p against the mean and the median:",
      [round(stats.ranksums(d18e["diff"].values, s18[:, j]).pvalue, 3) for j in (2, 3)])
print("main reading: rank sum tests significant at 0.05:", int((m6["rank sum p"] < 0.05).sum()), "of", len(m6),
      "; signed rank variant:", int((m6["signed rank p (variant)"] < 0.05).sum()), "of", len(m6))

**Result: reproduced.** None of the ten rank sum tests of the main reading is significant (p =
0.49 to 0.93), nor any under the other reading of the halves, nor the two Edge tests on the 18
files whose Edge is wholly on the screen (p = 0.70 and 0.95). The midpoint is above the block mean
or median by a median of -0.08 to 2.39 spikes/s. The paired signed rank test, shown as a variant,
finds 3 of the 10 main comparisons significant (S90 against the median, p = 0.025; S40 against the
mean, p = 0.015, and the median, p = 0.009); the claim is judged on the paper's rank sum test.

## fig7a-rms-contrast-fit

> Local RMS contrast along the Edge rises from 0.61 to 0.943 with luminance and is fit by an
> exponential model, f(l) = -0.34 exp(-0.05 l) + 0.95, with R2 = 0.9983.

In [ ]:
edge_recs = [r for r in aligned if r["folder"] == "Edge-compare"]
complete = [r for r in edge_recs if v.lum_at_fraction(r, v.folds(r)["Edge"], 0.95) >= 90]
print("Edge files with the whole Edge on the screen:", len(complete), "of", len(edge_recs))
EXP = {"lowest": 0.61, "highest": 0.943, "a": -0.34, "b": 0.05, "c": 0.95, "R2": 0.9983}
TOL = {"lowest": 0.061, "highest": 0.094, "a": 0.034, "b": 0.005, "c": 0.095, "R2": 0.0017}
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for a, reading in zip(ax, ["a", "b"]):
    curves = np.array([v.rms_contrast_profile(r, reading=reading)[1] for r in complete])
    l = v.rms_contrast_profile(complete[0], reading=reading)[0]
    curve = np.median(curves, 0)
    p, r2 = v.fit_exponential_lar(l, curve)
    got = {"lowest": np.nanmin(curve), "highest": np.nanmax(curve), "a": p[0], "b": p[1], "c": p[2], "R2": r2}
    print(f"reading ({reading}): spread over files at each step at most {np.nanmax(curves.max(0) - curves.min(0)):.3f}")
    print("  contrast at 0, 25, 50, 75, 100% luminance:", np.round(curve[[0, 17, 33, 50, 66]], 3))
    for k in EXP:
        print(f"  {k:8s} {got[k]:8.4f}  paper {EXP[k]:7.4f} ± {TOL[k]}  {'within' if abs(got[k] - EXP[k]) <= TOL[k] else 'outside'}")
    a.plot(l, curve, ".", label="computed"); a.plot(l, p[0] * np.exp(-p[1] * l) + p[2], "-", label="LAR fit")
    a.plot(l, -0.34 * np.exp(-0.05 * l) + 0.95, "--", label="paper's fit")
    a.set_title(f"reading ({reading}); range {np.nanmin(curve):.2f} to {np.nanmax(curve):.2f}"); a.set_xlabel("luminance (%)"); a.legend()
plt.tight_layout(); plt.show()

**Result: not reproduced.** Under neither definition of RMS contrast does the local contrast along
the Edge rise from 0.61 to 0.943. With the standard deviation over the mean (a) it is 1.98 at 0%,
near 0.86 from 25% to 75% and 1.28 at 100%; with the standard deviation of luminance scaled to 0
to 1 (b) it rises from 0.013 to 0.449. The fitted parameters are outside tolerance under both
(only c under (a) and R2 under (b) are within). The stimulus image holds luminance in percent with
black at 0. The text gives the display luminance at one level, 14.07 cd/m2 at 50% (Methods, Square
and edge), but not the black level, and a contrast that saturates near 0.95 would need a black
level above zero; no such level was tried here, as neither the text nor the files give one.

## fig7c-fold-edge-equivalence

> Neural responses to P180 are roughly equivalent to those to a 50% contrast Edge, while responses
> to S40 are close to those to a 90% contrast Edge, so sharper corners produce responses
> equivalent to higher contrasts along a straight edge.

In [ ]:
rms_levels = 0.610 + 0.037 * np.arange(10)
def lum_of_rms(c, a=-0.34, b=0.05, c0=0.95):
    return -np.log((c - c0) / a) / b
lum_rms = lum_of_rms(rms_levels)
lum_10 = np.arange(0, 100.1, 10)
print("luminance (%) of the ten RMS levels by the paper's fit:", np.round(lum_rms, 1))
# Edge responses from the 18 files whose Edge is wholly on the screen, as the Methods say it always was
# (Methods, Square and edge); all 22 are shown below as a reading
edge_used = [r for r in edge_recs if r["code"] in complete_codes]
print("Edge responses from", len(edge_used), "of", len(edge_recs), "Square and Edge files")
edge_rms = np.array([v.edge_response(r, r["smoothed"], v.baseline_value(r), lum_rms) for r in edge_used])
edge_10 = np.array([v.edge_response(r, r["smoothed"], v.baseline_value(r), lum_10) for r in edge_used])
edge_1 = np.array([v.edge_response(r, r["smoothed"], v.baseline_value(r), np.arange(0, 100.1, 1)) for r in edge_used])
edge_rms_all = np.array([v.edge_response(r, r["smoothed"], v.baseline_value(r), lum_rms) for r in edge_recs])
print("Edge response at 1% luminance steps (mean over neurons), 70% to 95%:",
      dict(zip(range(70, 96), np.round(edge_1.mean(0)[70:96], 2))))
fold_mean = fig4b_main["mean"]
print("fold means:", fold_mean.round(2).to_dict())
print("(a) Edge at 10% luminance steps:", dict(zip(lum_10.astype(int), np.round(edge_10.mean(0), 2))))
print("(b) Edge at the RMS levels:", dict(zip(np.round(rms_levels * 100, 1), np.round(edge_rms.mean(0), 2))))
res = {}
for f in ["P180", "S40", "S90", "S52"]:
    na = lum_10[np.argmin(np.abs(edge_10.mean(0) - fold_mean[f]))]
    nb = rms_levels[np.argmin(np.abs(edge_rms.mean(0) - fold_mean[f]))] * 100
    res[f] = (na, nb)
    print(f"{f}: nearest Edge level (a) {na:.0f}% luminance, (b) RMS x 100 = {nb:.1f}")
ok_a = res["P180"][0] in (40, 50, 60) and res["S40"][0] in (80, 90, 100)
ok_b = abs(res["P180"][1] - 61.0) < 0.05 and abs(res["S40"][1] - 90) <= 3.7
ok_b_level = abs(res["P180"][1] - 61.0) < 0.05 and abs(res["S40"][1] - 90.6) <= 3.7 + 0.05
print("reading (a) P180 within one step of 50% and S40 within one step of 90%:", ok_a)
print("reading (b) P180 nearest the lowest level and S40 within 3.7 of 90 [main]:", ok_b)
print("reading (b), one level from the level nearest 90 (90.6) [variant]:", ok_b_level)

def fig7c_eval(df, recs_, label):
    """Nearest Edge levels for P180 and S40 under a baseline reading (fold means from df, Edge from recs_)."""
    fm = df[df.fold.isin(folds4)].groupby("fold")["diff"].mean()
    e10 = np.array([v.edge_response(r, r["smoothed"], v.baseline_value(r, "smoothed", *READING_ARGS[label]), lum_10) for r in recs_]).mean(0)
    erm = np.array([v.edge_response(r, r["smoothed"], v.baseline_value(r, "smoothed", *READING_ARGS[label]), lum_rms) for r in recs_]).mean(0)
    na = {f: lum_10[np.argmin(np.abs(e10 - fm[f]))] for f in ("P180", "S40")}
    nb = {f: rms_levels[np.argmin(np.abs(erm - fm[f]))] * 100 for f in ("P180", "S40")}
    print(f"  {label}: P180 {fm['P180']:.2f}, S40 {fm['S40']:.2f}; Edge {e10.min():.2f} to {e10.max():.2f}; "
          f"(a) nearest P180 {na['P180']:.0f}%, S40 {na['S40']:.0f}%; (b) P180 {nb['P180']:.1f}, S40 {nb['S40']:.1f}; "
          f"holds (a) {na['P180'] in (40, 50, 60) and na['S40'] in (80, 90, 100)}, (b) {abs(nb['P180'] - 61) < 0.05 and abs(nb['S40'] - 90) <= 3.7}")
READING_ARGS = {"matched [main]": ("matched",), "centred": ("centred",), "inside": ("inside",), "files' BaseActivity": ("files",),
                "matched, Edge from all 22 files": ("matched",)}
print("other baselines, and the Edge from all 22 Square and Edge files:")
for label in READING_ARGS:
    if label == "matched, Edge from all 22 files":
        fig7c_eval(main, edge_recs, label)
    elif label == "files' BaseActivity":
        fig7c_eval(READINGS[label], [r for r in edge_used if np.isfinite(r["base_activity"])], label)
    else:
        fig7c_eval(READINGS[label], edge_used, label)
# variant: inverse of a refit to reading (a) of fig7a is not usable (the contrast is not monotonic); shown with the 10% profile only
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(lum_10, edge_10.mean(0), edge_10.std(0, ddof=1) / np.sqrt(len(edge_10)), fmt="o-", label="Edge, 10% steps")
ax.plot(lum_rms, edge_rms.mean(0), "gs", label="Edge, RMS levels")
for f in folds4:
    ax.axhline(fold_mean[f], ls=":", color="k"); ax.text(101, fold_mean[f], f, fontsize=8)
ax.set_xlabel("Edge luminance (%)"); ax.set_ylabel("spike rate difference (spikes/s)"); ax.legend(fontsize=8); plt.show()
print("Edge profile range", edge_10.mean(0).min().round(2), "to", edge_10.mean(0).max().round(2), "spikes/s")

**Result: not reproduced, borderline.** With the Edge from the 18 files whose Edge is wholly on
the screen, the Edge response ranges from 9.23 spikes/s (at 0% luminance) to 21.26 (at 80%), so the
P180 mean (4.03) lies below the whole Edge profile and is nearest the 0% level, not within one step
of 50% (reading a); S40 (26.27) is nearest 80%, within one step of 90%. Reading (a) fails clearly.
Under reading (b) P180 is nearest the lowest RMS level, which the rule accepts, and S40 is nearest
the 94.3 level. Whether that is "within one step of 90%" turns on how the step is counted, and the
criterion fixes neither way: as a distance from 90, 94.3 is 4.3 away against a step of 3.7, a miss
of 0.6, and (b) fails; as one level away from the level nearest 90 (90.6), the variant printed
above, (b) passes. The two readings are equally literal. This attempt takes the distance, an
assumption on which the outcome rests; under the level reading the claim would be reproduced.
Every other baseline reading and the Edge from all 22 files give the same under (a) (P180 nearest
0%) and under (b) (S40 nearest 94.3). The RMS levels are placed with the paper's fitted parameters,
as the claims file says, since fig7a-rms-contrast-fit could not reproduce them.

## fig7c-fold-edge-significance

> Responses to P180 are significantly lower than responses to a 94% contrast Edge, and responses
> to S40 are significantly higher than responses to an Edge of less than 68% contrast.

In [ ]:
fold_vals = {f: main.loc[main.fold == f, "diff"].values for f in folds4}
rows = []
for f in folds4:
    for j, c in enumerate(rms_levels):
        t = stats.ttest_ind(fold_vals[f], edge_rms[:, j])
        rows.append({"fold": f, "rms": round(c, 3), "mean difference": fold_vals[f].mean() - edge_rms[:, j].mean(), "t": t.statistic, "p": t.pvalue})
t7 = pd.DataFrame(rows)
t7["(a) BH over 40"] = multipletests(t7.p, method="fdr_bh")[1]
t7["(b) BH per fold"] = t7.groupby("fold")["p"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
t7["(c) Bonferroni over 40"] = np.minimum(t7.p * 40, 1)
t7["(c) Bonferroni per fold"] = np.minimum(t7.p * 10, 1)
print(t7[t7.fold.isin(["P180", "S40"])].to_string(index=False, float_format=lambda x: f"{x:.4g}"))
for col in ["(a) BH over 40", "(b) BH per fold", "(c) Bonferroni over 40", "(c) Bonferroni per fold"]:
    p180 = t7[(t7.fold == "P180") & (t7.rms == 0.943)].iloc[0]
    s40 = t7[(t7.fold == "S40") & (t7.rms.isin([0.61, 0.647]))]
    ok = p180[col] < 0.05 and p180["mean difference"] < 0 and (s40[col] < 0.05).all() and (s40["mean difference"] > 0).all()
    sig = t7[(t7.fold == "S40") & (t7[col] < 0.05) & (t7["mean difference"] > 0)].rms.tolist()
    print(f"{col}: claim holds {ok}; S40 significantly higher at RMS {sig}; P180 lower at 0.943: p = {p180[col]:.3g}")
# variant: the Edge from all 22 Square and Edge files
t22 = pd.DataFrame([{"fold": f, "rms": round(c, 3), "mean difference": fold_vals[f].mean() - edge_rms_all[:, j].mean(),
                     "p": stats.ttest_ind(fold_vals[f], edge_rms_all[:, j]).pvalue} for f in folds4 for j, c in enumerate(rms_levels)])
t22["(a) BH over 40"] = multipletests(t22.p, method="fdr_bh")[1]
t22["(b) BH per fold"] = t22.groupby("fold")["p"].transform(lambda p: multipletests(p, method="fdr_bh")[1])
for col in ["(a) BH over 40", "(b) BH per fold"]:
    p180 = t22[(t22.fold == "P180") & (t22.rms == 0.943)].iloc[0]
    s40 = t22[(t22.fold == "S40") & (t22.rms.isin([0.61, 0.647]))]
    ok = p180[col] < 0.05 and p180["mean difference"] < 0 and (s40[col] < 0.05).all() and (s40["mean difference"] > 0).all()
    print(f"Edge from all 22 files, {col}: claim holds {ok}; P180 lower at 0.943: p = {p180[col]:.3g}; S40 at 0.610 and 0.647: p =",
          np.round(s40[col].values, 4).tolist(), "; mean differences", np.round(s40["mean difference"].values, 2).tolist())


**Result: reproduced.** With the Edge from the 18 files whose Edge is wholly on the screen, under
BH correction over all 40 tests (a) P180 is significantly lower than the Edge at RMS 0.943
(adjusted p = 0.015) and S40 significantly higher at 0.610 and 0.647 (both 0.033); BH per fold (b)
gives the same (0.0038; 0.022 and 0.022), and so does Bonferroni per fold (0.0076; 0.033 and
0.045). Bonferroni over 40 tests fails (S40 at 0.610, p = 0.13). With the Edge from all 22 files
the claim holds under (a) and (b). Under (a) S40 is also significantly higher than the Edge at
0.684, 0.721 and 0.758, beyond the paper's "less than 68%", and P180 is lower than the Edge at
every level from 0.647 up.

## fig7d-nonlinear

> The relationship between corner angle and V1 spike rate difference is parametric but apparently
> not linear; the paper says it cannot be certain of this, given the few angles tested.

In [ ]:
ang = np.array([40, 52, 90, 180.0])
mean = fig4b_main.loc[["S40", "S52", "S90", "P180"], "mean"].values
sem = fig4b_main.loc[["S40", "S52", "S90", "P180"], "sem"].values
slope, icpt = np.polyfit(ang, mean, 1)
resid = mean - (slope * ang + icpt)
print("line:", round(slope, 4), "spikes/s per degree, intercept", round(icpt, 2))
print(pd.DataFrame({"angle": ang, "mean": mean, "sem": sem, "residual": resid, "|residual| > SEM": np.abs(resid) > sem}).round(3).to_string(index=False))
monotonic = bool(np.all(np.diff(mean) < 0))
print("monotonic:", monotonic, "; non-linear by the SEM rule:", bool(np.any(np.abs(resid) > sem)))
sl = np.diff(mean) / np.diff(ang); sl_se = np.sqrt(sem[1:] ** 2 + sem[:-1] ** 2) / np.diff(ang)
print("variant, slopes between successive angles:", np.round(sl, 3), "± (propagated)", np.round(sl_se, 3))
print("  successive slopes differ by more than their combined SE:", [bool(abs(sl[i + 1] - sl[i]) > np.hypot(sl_se[i], sl_se[i + 1])) for i in range(2)])
print("other baselines (monotonic; folds with |residual| > SEM):")
for label, df in list(READINGS.items())[1:]:
    g = df[df.fold.isin(folds4)].groupby("fold")["diff"]
    m_, se_ = g.mean().reindex(["S40", "S52", "S90", "P180"]).values, g.sem().reindex(["S40", "S52", "S90", "P180"]).values
    r_ = m_ - np.polyval(np.polyfit(ang, m_, 1), ang)
    print(f"  {label}: monotonic {bool(np.all(np.diff(m_) < 0))}; off the line by more than the SEM:",
          [f for f, x, e in zip(["S40", "S52", "S90", "P180"], r_, se_) if abs(x) > e])

**Result: reproduced.** The four means decrease monotonically with angle, and three lie off the
least-squares line (-0.142 spikes/s per degree) by more than their SEM (S40 by 4.63 against 4.29,
S90 by -6.15 against 2.22, P180 by 2.26 against 1.18), which the claims file's rule, an
assumption, reads as non-linear; S90 and P180 lie off it under every other baseline reading shown.
In the variant the slope from 52 to 90 degrees (-0.284 spikes/s per degree) differs from that from
90 to 180 (-0.049) by more than their combined SE, while the slopes from 40 to 52 (-0.59 ± 0.44)
and from 52 to 90 do not differ.

## Blocked: fig7d-dog-similar, s3-latency-peak

fig7d-dog-similar needs the output of the difference-of-Gaussians model of reference [3], which
the paper shows only as a line in Fig 7D; it names no deposited data, and the data set holds none.
s3-latency-peak needs spike times and eye positions to rebuild the maps at other latencies; the
files hold only maps made at 60 ms.

In [ ]:
print("total run time", round((time.time() - T0) / 60, 1), "min; peak memory", round(resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6, 2), "GB")

## Summary

| Claim | Outcome |
|---|---|
| neuron-counts | not reproduced |
| spikes-per-condition | blocked: data missing |
| s4-baseline-no-difference | not reproduced (rests on an assumption) |
| fig4a-above-baseline | reproduced (rests on an assumption) |
| fig4b-spike-rate-difference | partially reproduced (rests on an assumption) |
| fig4b-pairwise-significance | reproduced |
| fig5-average-maps | reproduced (rests on an assumption) |
| table1-edge | reproduced (rests on an assumption) |
| table1-p180 | partially reproduced (rests on an assumption) |
| table1-s90 | not reproduced (rests on an assumption) |
| table1-s52 | reproduced (rests on an assumption) |
| table1-s40 | reproduced (rests on an assumption) |
| fig6-midpoint-representative | reproduced |
| fig7a-rms-contrast-fit | not reproduced |
| fig7c-fold-edge-equivalence | not reproduced (rests on an assumption) |
| fig7c-fold-edge-significance | reproduced (rests on an assumption) |
| fig7d-nonlinear | reproduced (rests on an assumption) |
| fig7d-dog-similar | blocked: data missing |
| s3-latency-peak | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.**

- It uses the maps the files hold, already made by the authors at 60 ms latency and smoothed,
  in the authors' ROI frame (`Standard`). It cannot redo the eye-signal correction or the
  registration, and it cannot use the 15 Star recordings whose files lack that frame, so the
  Star claims rest on 12 neurons (the paper has 30), Corners on 26 (28) and Square and Edge on
  22 (25).
- The Corners response is taken at the bright S40 fold, as the claims file says since a reader
  amended it in answer to this attempt's questions.
- The text does not say where the baseline squares are. The main reading takes, per stimulus, the
  placement closest to the files' own `BaseActivity`: centred on the corners of the circumscribing
  square for Square and Edge (read from the screen maps, as these squares run past the authors'
  frame), inside the corners for the Star, and the bottom corners of the ROI for Corners, where no
  placement tried matches closely. The other placements and the files' values are shown; the
  baseline test across stimuli changes outcome with them, and Table 1 counts change.
- The Edge measures use the 18 of 22 Square and Edge files whose Edge is wholly on the screen, as
  the Methods say it always was; all 22 are shown.
- RMS contrast is computed from the stimulus image in percent luminance; the paper's
  definition (its reference 21) and the display's black level are not in the text or the files.

**What was found about the dataset.**

- The files hold 53 neurons, not the paper's 58 (the spreadsheet lists 58, five with no data),
  and fewer per condition (26 Corners, 27 Star, 22 Square and Edge against 28, 30, 25).
- Two files in the `Star` folder (`ABS042700J2`, `ABS051000J3`) are Corners-type recordings; the
  spreadsheet marks both neurons as tested with the Star.
- The descriptor's field list is wrong: `image` is a spike rate in spikes per 1 ms sample, not a
  count; most files hold an undocumented `Standard` struct with the authors' aligned maps,
  polygons and in some a `BaseActivity`; `smoothed_image_norm` is absent from 39 of 77 files and,
  where present, is `smoothed_image` minus `BaseActivity` in 5, a linear rescaling of it in 21 and
  in 11 not a linear function of it, with 26% to 77% of
  the pixels at exactly -1 (data check).
- The maps are in gaze coordinates, and the receptive-field offset needed to place them on the
  stimulus is in no file; 15 Star recordings cannot be aligned for that reason.
- No file holds spike times, eye positions or (but for one) the eye time map, so the spike counts
  per condition and the latency analysis cannot be repeated.
- In four Square and Edge files the 100% end of the Edge is cut by the screen border, which the
  Methods say never happened.
- The authors' Square and Edge frame (151 DU wide) is too narrow to hold the baseline squares
  centred on the corners of the circumscribing square, which the files' `BaseActivity` matches best.